In [38]:
import json

def expand_rare_labels(recs, lookback=1):
    """
    For each jump/flip occurrence, also set that label =1 on the up-to-`lookback`
    preceding frames, but only where that frame's can_jump (obs[-1]) is 1.
    Mutates and returns recs. Widens rare labels ~4x for BC.
    """
    n = len(recs)
    # collect original occurrence indices first (so we don't chain off added labels)
    jump_hits = [i for i, r in enumerate(recs) if r["action"]["jump"]]
    flip_hits = [i for i, r in enumerate(recs) if r["action"]["flip"]]

    def spread(hits, key):
        for i in hits:
            for k in range(max(0, i - lookback), i):     # the frames before i
                if recs[k]["obs"][-1] >= 0.5:            # can_jump true on that frame
                    recs[k]["action"][key] = 1

    spread(jump_hits, "jump")
    spread(flip_hits, "flip")
    return recs

In [ ]:
import glob, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

ACTION_KEYS = ["left", "right", "up", "down", "boost", "jump", "flip"]
RARE = ["jump", "flip"]                      # heads to up-weight
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


def load_sessions(pattern="data/*.jsonl", shift=1, lookback=1):
    obs, act = [], []
    for path in sorted(glob.glob(pattern)):
        recs = []
        for l in open(path):
            l = l.strip()
            if not l: continue
            try:
                recs.append(json.loads(l))
            except json.JSONDecodeError:
                continue                       # skip corrupt lines (you had 1)
        recs = expand_rare_labels(recs, lookback=lookback)   # <-- widen jump/flip

        for t in range(len(recs) - shift):
            obs.append(recs[t]["obs"])
            act.append([recs[t + shift]["action"][k] for k in ACTION_KEYS])
    obs = np.array(obs, dtype=np.float32)
    act = np.array(act, dtype=np.float32)

    # (the impossible-jump filter can stay, but it's now redundant since
    #  expand only adds labels where can_jump=1; keep as a safety net)
    canjump = obs[:, -1]
    ji, fi = ACTION_KEYS.index("jump"), ACTION_KEYS.index("flip")
    bad = ((act[:, ji] + act[:, fi]) > 0) & (canjump < 0.5)
    act[bad, ji] = 0; act[bad, fi] = 0

    print(f"{len(obs)} pairs, obs dim {obs.shape[1]}, zeroed {int(bad.sum())} impossible")
    for i, k in enumerate(ACTION_KEYS):
        print(f"  {k:6s}: {int(act[:,i].sum())} ({100*act[:,i].mean():.1f}%)")
    return obs, act


# ---------- normalization ----------
def fit_normalizer(obs):
    mean = obs.mean(0); std = obs.std(0); std[std < 1e-6] = 1.0
    return mean, std


class BCData(Dataset):
    def __init__(self, obs, act):
        self.obs = torch.from_numpy(obs)
        self.act = torch.from_numpy(act)
    def __len__(self): return len(self.obs)
    def __getitem__(self, i): return self.obs[i], self.act[i]


# ---------- model: shared trunk, per-action heads ----------
class BCPolicy(nn.Module):
    def __init__(self, in_dim, n_actions=7, hidden=128):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        # one logit per action (independent sigmoids = multi-label)
        self.head = nn.Linear(hidden, n_actions)
    def forward(self, x):
        return self.head(self.trunk(x))       # logits


# ---------- training ----------
def train_bc(pattern="data/*.jsonl", epochs=25, batch=128, lr=1e-3,
             rare_weight_cap=50.0, val_frac=0.15):
    obs, act = load_sessions(pattern)
    norm = fit_normalizer(obs)
    mean, std = norm
    obs = (obs - mean) / std

    # train/val split
    n = len(obs); idx = np.random.permutation(n); cut = int(n * (1 - val_frac))
    tr, va = idx[:cut], idx[cut:]
    tr_ds, va_ds = BCData(obs[tr], act[tr]), BCData(obs[va], act[va])

    # per-action positive weight = neg/pos, capped so rare classes don't explode
    pos = act[tr].sum(0); neg = len(tr) - pos
    pos_weight = np.minimum(neg / np.maximum(pos, 1.0), rare_weight_cap).astype(np.float32)
    print("pos_weight:", dict(zip(ACTION_KEYS, pos_weight.round(1))))
    pos_weight_t = torch.tensor(pos_weight, device=DEVICE)

    # oversample frames containing a rare action so the batch sees them often
    ji, fi = ACTION_KEYS.index("jump"), ACTION_KEYS.index("flip")
    has_rare = (act[tr][:, ji] + act[tr][:, fi]) > 0
    sample_w = np.where(has_rare, 20.0, 1.0)          # 20x weight for rare frames
    sampler = WeightedRandomSampler(sample_w, num_samples=len(tr), replacement=True)

    tr_loader = DataLoader(tr_ds, batch_size=batch, sampler=sampler)
    va_loader = DataLoader(va_ds, batch_size=batch)

    model = BCPolicy(obs.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)

    for ep in range(epochs):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward(); opt.step()

        # validation: per-action recall/precision on rare heads
        if (ep + 1) % 5 == 0 or ep == epochs - 1:
            model.eval()
            with torch.no_grad():
                P = torch.sigmoid(model(torch.from_numpy(obs[va]).to(DEVICE))).cpu().numpy()
            Y = act[va]
            msg = []
            print(ep)
            for k in RARE:
                i = ACTION_KEYS.index(k)
                pred = (P[:, i] > 0.5).astype(float)
                tp = ((pred == 1) & (Y[:, i] == 1)).sum()
                fp = ((pred == 1) & (Y[:, i] == 0)).sum()
                fn = ((pred == 0) & (Y[:, i] == 1)).sum()
                rec = tp / max(tp + fn, 1); prec = tp / max(tp + fp, 1)
                msg.append(f"{k} rec={rec:.2f} prec={prec:.2f} (tp{int(tp)} fp{int(fp)} fn{int(fn)})")
            print(f"epoch {ep+1}: " + " | ".join(msg))
    return model, norm

my_bc_model, norm = train_bc()

5261 pairs, obs dim 41, zeroed 19 impossible
  left  : 2428 (46.2%)
  right : 2008 (38.2%)
  up    : 784 (14.9%)
  down  : 493 (9.4%)
  boost : 2402 (45.7%)
  jump  : 27 (0.5%)
  flip  : 115 (2.2%)
pos_weight: {'left': np.float32(1.2), 'right': np.float32(1.6), 'up': np.float32(5.7), 'down': np.float32(9.7), 'boost': np.float32(1.2), 'jump': np.float32(50.0), 'flip': np.float32(44.2)}
4
epoch 5: jump rec=1.00 prec=0.01 (tp4 fp280 fn0) | flip rec=1.00 prec=0.03 (tp16 fp463 fn0)
9
epoch 10: jump rec=1.00 prec=0.02 (tp4 fp158 fn0) | flip rec=1.00 prec=0.04 (tp16 fp359 fn0)
14
epoch 15: jump rec=1.00 prec=0.04 (tp4 fp96 fn0) | flip rec=1.00 prec=0.05 (tp16 fp277 fn0)
19
epoch 20: jump rec=1.00 prec=0.07 (tp4 fp57 fn0) | flip rec=0.94 prec=0.06 (tp15 fp224 fn1)
24
epoch 25: jump rec=1.00 prec=0.07 (tp4 fp54 fn0) | flip rec=0.94 prec=0.07 (tp15 fp213 fn1)


In [ ]:
version = "25_5k_lookback1"
import os
def save_bc(model, norm, path=f"models/{version}.pt"):
    """Save BC weights + normalizer + architecture metadata in one file."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    mean, std = norm
    torch.save({
        "state_dict": model.state_dict(),
        "mean": np.asarray(mean, dtype=np.float32),
        "std":  np.asarray(std,  dtype=np.float32),
        "in_dim": model.trunk[0].in_features,
        "n_actions": model.head.out_features,
        "action_keys": ACTION_KEYS,
    }, path)
    print(f"saved BC policy -> {path}")

save_bc(my_bc_model, norm)

saved BC policy -> models/25_5k_lookback1.pt


In [ ]:
def load_bc(path=f"models/{version}.pt", device=None):
    """Rebuild the model + normalizer from disk. Returns (model, (mean, std))."""
    device = device or DEVICE
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model = BCPolicy(ckpt["in_dim"], ckpt["n_actions"]).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    norm = (ckpt["mean"], ckpt["std"])
    print(f"loaded BC policy <- {path}  (in_dim={ckpt['in_dim']})")
    return model, norm

In [ ]:
my_model, norm = load_bc()                     # ready for run_bot(my_model, norm)
my_model.eval()
import numpy as np
obs, act = load_sessions()
mean, std = norm
obsn = (obs - mean) / std
with torch.no_grad():
    P = torch.sigmoid(my_model(torch.from_numpy(obsn).to(DEVICE))).cpu().numpy()
for i, k in enumerate(ACTION_KEYS):
    pred = (P[:, i] > 0.5).astype(float)
    tp = ((pred==1)&(act[:,i]==1)).sum(); fp=((pred==1)&(act[:,i]==0)).sum()
    fn=((pred==0)&(act[:,i]==1)).sum()
    rec = tp/max(tp+fn,1); prec = tp/max(tp+fp,1)
    print(f"{k:6s} rec={rec:.2f} prec={prec:.2f}")

loaded BC policy <- models/25_5k_lookback1.pt  (in_dim=41)
5261 pairs, obs dim 41, zeroed 19 impossible
  left  : 2428 (46.2%)
  right : 2008 (38.2%)
  up    : 784 (14.9%)
  down  : 493 (9.4%)
  boost : 2402 (45.7%)
  jump  : 27 (0.5%)
  flip  : 115 (2.2%)
left   rec=0.82 prec=0.70
right  rec=0.75 prec=0.75
up     rec=0.75 prec=0.35
down   rec=0.85 prec=0.32
boost  rec=0.72 prec=0.63
jump   rec=1.00 prec=0.08
flip   rec=0.99 prec=0.07
